# 21.7 Answer-set programming

Answer-set programs describe constraints on possible worlds, then keep the worlds that are self-consistent and supported.

ASP generates candidate worlds, rejects constraint violations, and keeps self-supporting stable models. It is useful for scheduling, configuration, planning, and repair-set search.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random

import matplotlib.pyplot as plt

random.seed(2107)


def powerset(items):
    names = list(items)
    for values in itertools.product([False, True], repeat=len(names)):
        yield {name for name, value in zip(names, values) if value}


def body_true(rule, world):
    positive = set(rule.get("pos", []))
    negative = set(rule.get("not", []))
    return positive.issubset(world) and world.isdisjoint(negative)


def positive_closure(rules):
    closure = set()
    changed = True
    while changed:
        changed = False
        for rule in rules:
            if set(rule.get("pos", [])).issubset(closure):
                head = rule.get("head")
                if head is not None and head not in closure:
                    closure.add(head)
                    changed = True
    return closure


def stable_models(atoms, rules, constraints, costs):
    models = []
    candidates = 0
    rejected = []
    for world in powerset(atoms):
        candidates += 1
        violated = [constraint for constraint in constraints if body_true(constraint, world)]
        if violated:
            rejected.append((world, "constraint"))
            continue
        reduct = []
        for rule in rules:
            if world.isdisjoint(set(rule.get("not", []))):
                reduct.append({"head": rule.get("head"), "pos": list(rule.get("pos", []))})
        least_model = positive_closure(reduct)
        if least_model != world:
            rejected.append((world, "stable-reduct"))
            continue
        cost = sum(costs.get(atom, 0) for atom in world)
        models.append({"world": world, "cost": cost})
    best = min(models, key=lambda item: item["cost"]) if models else None
    return {"models": models, "best": best, "candidates": candidates, "rejected": rejected}


def make_asp_ladder():
    return [
        {"name": "D1 choose a or b", "atoms": ["a", "b"], "rules": [{"head": "a", "pos": [], "not": ["b"]}, {"head": "b", "pos": [], "not": ["a"]}], "constraints": [{"pos": ["a", "b"], "not": []}], "costs": {"a": 2, "b": 1}, "expected": 2},
        {"name": "D2 small schedule", "atoms": ["ann_am", "ann_pm", "bo_am", "bo_pm"], "rules": [{"head": "ann_am", "pos": [], "not": ["ann_pm"]}, {"head": "ann_pm", "pos": [], "not": ["ann_am"]}, {"head": "bo_am", "pos": [], "not": ["bo_pm"]}, {"head": "bo_pm", "pos": [], "not": ["bo_am"]}], "constraints": [{"pos": ["ann_am", "ann_pm"], "not": []}, {"pos": ["bo_am", "bo_pm"], "not": []}], "costs": {"ann_am": 1, "ann_pm": 2, "bo_am": 2, "bo_pm": 1}, "expected": 4},
        {"name": "D3 defaults conflicts", "atoms": ["go", "stay", "rain"], "rules": [{"head": "go", "pos": [], "not": ["stay", "rain"]}, {"head": "stay", "pos": [], "not": ["go"]}, {"head": "rain", "pos": ["rain"], "not": []}], "constraints": [{"pos": ["go", "stay"], "not": []}], "costs": {"go": 1, "stay": 2, "rain": 0}, "expected": 2},
        {"name": "D4 configuration", "atoms": ["small", "large", "red", "blue", "wifi"], "rules": [{"head": "small", "pos": [], "not": ["large"]}, {"head": "large", "pos": [], "not": ["small"]}, {"head": "red", "pos": [], "not": ["blue"]}, {"head": "blue", "pos": [], "not": ["red"]}, {"head": "wifi", "pos": ["large"], "not": []}], "constraints": [{"pos": ["small", "large"], "not": []}, {"pos": ["red", "blue"], "not": []}, {"pos": ["small", "wifi"], "not": []}], "costs": {"small": 1, "large": 3, "red": 1, "blue": 1, "wifi": 1}, "expected": 4},
        {"name": "D5 combinatorial schedule", "atoms": ["ann_am", "ann_pm", "bo_am", "bo_pm", "cy_am", "cy_pm", "lead_ann", "lead_bo"], "rules": [{"head": "ann_am", "pos": [], "not": ["ann_pm"]}, {"head": "ann_pm", "pos": [], "not": ["ann_am"]}, {"head": "bo_am", "pos": [], "not": ["bo_pm"]}, {"head": "bo_pm", "pos": [], "not": ["bo_am"]}, {"head": "cy_am", "pos": [], "not": ["cy_pm"]}, {"head": "cy_pm", "pos": [], "not": ["cy_am"]}, {"head": "lead_ann", "pos": ["ann_am"], "not": ["lead_bo"]}, {"head": "lead_bo", "pos": ["bo_am"], "not": ["lead_ann"]}], "constraints": [{"pos": ["ann_am", "ann_pm"], "not": []}, {"pos": ["bo_am", "bo_pm"], "not": []}, {"pos": ["cy_am", "cy_pm"], "not": []}, {"pos": ["ann_pm", "bo_pm", "cy_pm"], "not": []}, {"pos": ["lead_ann", "lead_bo"], "not": []}], "costs": {"ann_am": 2, "ann_pm": 1, "bo_am": 1, "bo_pm": 2, "cy_am": 2, "cy_pm": 1, "lead_ann": 1, "lead_bo": 2}, "expected": 9},
    ]


## The concept, built once (D1)

For atoms $\{a,b\}$ there are $4$ subsets. Choice rules keep $\{a\}$ and $\{b\}$, the not-both constraint rejects $\{a,b\}$, and optimization compares $cost(\{a\})=2$ with $cost(\{b\})=1$.

In [ ]:

atoms = ["a", "b"]
rules = [{"head": "a", "pos": [], "not": ["b"]}, {"head": "b", "pos": [], "not": ["a"]}]
constraints = [{"pos": ["a", "b"], "not": []}]
costs = {"a": 2, "b": 1}

d1 = stable_models(atoms, rules, constraints, costs)
print("candidates", d1["candidates"])
print("models", d1["models"])
print("best", d1["best"])


The lesson numbers are exact: $4$ candidate worlds, $2$ stable choices, and the best model is $\{b\}$ with cost $1$ rather than $2$.

In [ ]:

assert d1["candidates"] == 4
assert len(d1["models"]) == 2
assert d1["best"]["world"] == {"b"}
assert d1["best"]["cost"] == 1


## The dataset ladder

The ladder grows from a two-atom program to a combinatorial schedule with optimization, support checks, and multiple constraints.

In [ ]:

asp_ladder = make_asp_ladder()

for rung in asp_ladder:
    print(rung["name"], "atoms", len(rung["atoms"]), "candidates", 2 ** len(rung["atoms"]), "rules", len(rung["rules"]))


## Run the same method across D1-D5

Every rung uses the same enumerating ASP checker. The metric records stable-model correctness, candidate worlds, rejections, and best cost.

In [ ]:

asp_results = []
for rung in asp_ladder:
    result = stable_models(rung["atoms"], rung["rules"], rung["constraints"], rung["costs"])
    correct = len(result["models"]) == rung["expected"]
    best_cost = result["best"]["cost"] if result["best"] is not None else None
    asp_results.append({
        "name": rung["name"],
        "atoms": len(rung["atoms"]),
        "candidates": result["candidates"],
        "models": len(result["models"]),
        "rejected": len(result["rejected"]),
        "best_cost": best_cost,
        "correct": correct,
        "result": result,
    })

for row in asp_results:
    print(row["name"], row["atoms"], row["candidates"], row["models"], row["best_cost"], row["correct"])


## Results visualization

The panels show answer sets and support outcomes. The curve shows candidates pruned and best cost as size rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], asp_results):
    text = "\n".join(str(model) for model in row["result"]["models"][:5])
    ax.text(0.03, 0.95, text, va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["atoms"] for row in asp_results], [row["candidates"] for row in asp_results], marker="o", label="candidates")
ax.plot([row["atoms"] for row in asp_results], [row["rejected"] for row in asp_results], marker="x", label="rejected")
ax.set_xlabel("atoms")
ax.set_title("candidate pruning")
ax.legend()
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: confusing a surviving candidate with a proved answer. D5 must pass support and every constraint before a schedule counts as stable.

In [ ]:

hard = asp_ladder[-1]
naive_candidate = {"ann_am", "bo_am", "cy_am"}
naive_survives = len(naive_candidate) > 0
fixed = stable_models(hard["atoms"], hard["rules"], hard["constraints"], hard["costs"])
accepted_worlds = [model["world"] for model in fixed["models"]]

print("naive surviving candidate", naive_survives)
print("fixed accepts naive", naive_candidate in accepted_worlds)
print("checked candidates", fixed["candidates"])
print("best", fixed["best"])


## Evaluate it + Practice

- Metric: stable-model correctness, candidate worlds, and best cost, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add a fourth worker to D5 and predict the candidate-world count.

Practice: Remove the support rule for lead_bo and find which worlds disappear.

Practice: Add a hard constraint and compare rejected counts before and after.